# 🔴 KaizenStat — Advanced Demo (30 min)

**Level:** Advanced | **Time:** ~30 minutes | **Dataset:** Titanic

Covers: feature engineering, custom checks, drift detection, custom models,
hyperparameter tuning, feature impact, codegen, production export.

In [ ]:
!pip install kaizenstat -q
print("KaizenStat installed")

## 1. Feature Engineering (before fit)

`load()` gives us the raw DataFrame. We extract features from `Name` and `Cabin`
**before** `fit()` auto-drops them.

In [ ]:
import pandas as pd
from kaizenstat import DataDoctor

doctor = DataDoctor()
doctor.load("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

# Access the raw loaded frame for feature engineering
df = doctor._df.copy()

# Title from Name (before auto-drop)
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
rare = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
df['Title'] = df['Title'].replace(rare, 'Rare')
df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

# Deck from Cabin (before auto-drop)
df['Deck'] = df['Cabin'].str[0].fillna('Unknown')

# Family features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Reload enriched frame, then fit — auto-drops Name/Cabin/PassengerId/Ticket
doctor._df = df
doctor.fit(target="Survived")
print("Features:", [c for c in doctor._df.columns if c != 'Survived'])

## 2. Custom Validation Check

In [ ]:
def check_family(df, target):
    issues = []
    if 'FamilySize' in df.columns and df['FamilySize'].max() > 20:
        issues.append(f"Suspiciously large FamilySize: {df['FamilySize'].max()}")
    return issues

doctor.add_check(check_family, name="family_check")
doctor.validate()

## 3. Dataset Difficulty

In [ ]:
d = doctor.dataset_difficulty()
print(f"Difficulty: {d:.3f}  (0=trivial, 1=impossible)")

## 4. Drift Detection

In [ ]:
X = doctor._df.drop(columns=['Survived'])
drift = doctor.detect_drift(X.iloc[:600], X.iloc[600:])
if drift:
    print("Drift detected:")
    for feat, p in drift.items():
        print(f"  {feat}: p={p:.4f}")
else:
    print("No significant drift")

In [ ]:
doctor.fix(safe=True)

## 5. Add Custom Models

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier

doctor.add_model("SVM_RBF", SVC(probability=True, kernel='rbf', C=1.0))
doctor.add_model("ExtraTrees", ExtraTreesClassifier(n_estimators=100, random_state=42))

## 6. Train with Tuning

In [ ]:
result = doctor.train(cv=5, tune=True, n_iter=20)
print(f"Best model:  {result.model_name}")
print(f"Test score:  {result.test_score:.4f}")
if hasattr(result, 'best_params') and result.best_params:
    print(f"Best params: {result.best_params}")

## 7. Debug + Feature Impact

In [ ]:
debug = doctor.debug_model()
print(f"Gap: {debug.gap:.4f}")

impact = doctor.feature_impact(top_n=10)
print("\nTop features:")
for feat, drop in sorted(impact.items(), key=lambda x: -x[1])[:8]:
    bar = chr(9608) * max(1, int(drop * 200))
    print(f"  {feat:20s}  {drop:.4f}  {bar}")

In [ ]:
trust = doctor.trust_score()
print(f"Trust Score: {trust.score:.0f} / 100")
print(f"Pipeline Confidence: {doctor.pipeline_confidence()} / 100")

## 8. Report + Codegen

In [ ]:
doctor.report(output_path="advanced_report.html")
from IPython.display import IFrame, display
display(IFrame(src="advanced_report.html", width="100%", height="600px"))

In [ ]:
script = doctor.codegen(output_path="titanic_pipeline.py")
print(f"Script: {script}")
with open(script) as f:
    for i, line in enumerate(f):
        if i >= 30: break
        print(line, end='')

## 9. Export + Production Inference

In [ ]:
model_path = doctor.export_model(path="titanic_model.joblib")
print(f"Model exported: {model_path}")

In [ ]:
import joblib, pandas as pd

new_passenger = pd.DataFrame([{
    'Pclass': 1, 'Sex': 'female', 'Age': 29,
    'SibSp': 0, 'Parch': 0, 'Fare': 100.0, 'Embarked': 'S',
    'Title': 'Miss', 'Deck': 'C', 'FamilySize': 1, 'IsAlone': 1
}])

model = joblib.load(model_path)
pred = model.predict(new_passenger)[0]
proba = model.predict_proba(new_passenger)[0]

print(f"Prediction: {'Survived' if pred == 1 else 'Did not survive'}")
print(f"Confidence: {max(proba):.1%}")

---
## Quick Reference

```python
doctor = DataDoctor()
doctor.load("any_file.csv")         # CSV, Excel, Parquet, URL
doctor.fit(target="label")          # auto-drops ID columns
doctor.add_check(fn, name="check")
doctor.fix(safe=True)
doctor.add_model("SVM", SVC(...))
doctor.train(tune=True, n_iter=20)
doctor.feature_impact()
doctor.trust_score()
doctor.report()
doctor.codegen()
doctor.export_model()
```

---
*KaizenStat v0.5.1 · [GitHub](https://github.com/kaizenstat-python/KaizenStat) · MIT License*